# Fine-tune the federated-search query planner (Qwen3.8-27B · QLoRA · Unsloth)

This notebook trains the model that turns a user's question into a structured **`QueryPlan`**
(intent, which systems to search, canonical entities, rewritten query), and measures it **before and after** fine-tuning.

| Runtime | Profile | What it's for |
|---|---|---|
| **A100 40 GB** (Colab Pro) or better | `qwen3.8-27b-qlora` | the real run: 4-bit QLoRA of `unsloth/Qwen3.8-27B` (~24 GB VRAM, ~1–2 h) |
| T4 16 GB (free) | `qwen3.5-4b-smoke` | checks the whole pipeline end-to-end in ~20 min |

`PROFILE = "auto"` picks based on the GPU it finds. **Runtime → Change runtime type → GPU** before starting.

**Secrets** (🔑 icon in the left sidebar, all optional): `HF_TOKEN` to push the adapter to the Hugging Face Hub; `GH_TOKEN` only if you run this from a private fork.

Steps: clone repo → install → generate data → load model → **baseline eval** → train → **fine-tuned eval** → save adapter.

## 1. Settings

In [ ]:
PROFILE = "auto"          # "auto" | "qwen3.8-27b-qlora" | "qwen3.5-4b-smoke"
EVAL_LIMIT = 200          # test questions scored before/after (479 available); lower = faster
SAVE_TO_DRIVE = True      # copy adapter + eval report to Google Drive
HF_REPO = ""              # e.g. "your-username/fedsearch-qwen3.8-27b-planner" to push the adapter (needs HF_TOKEN)

## 2. Check the GPU

In [ ]:
import subprocess, torch
assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> GPU"
gpu = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
if PROFILE == "auto":
    PROFILE = "qwen3.8-27b-qlora" if vram_gb >= 30 else "qwen3.5-4b-smoke"
print(f"{gpu}: {vram_gb:.0f} GB  ->  profile {PROFILE}")
if PROFILE.startswith("qwen3.8-27b") and vram_gb < 23:
    raise SystemExit("27B QLoRA needs ~24 GB VRAM. Switch to an A100 runtime or use PROFILE='qwen3.5-4b-smoke'.")

## 3. Get the code

In [ ]:
import os, subprocess
REPO = "priyankavlt/platform-agnostic-federated-search-with-fine-tuned-llm"
BRANCH = "main"
WORKDIR = "/content/fedsearch"

def _secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

if not os.path.exists(WORKDIR):
    token = _secret("GH_TOKEN")  # only needed for a private fork
    url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, url, WORKDIR], check=True)
os.chdir(WORKDIR)
# An editable pip install isn't visible to an already-running kernel, so put the code on the path directly.
import sys
sys.path[:0] = [f"{WORKDIR}/src", WORKDIR]
os.environ["PYTHONPATH"] = f"{WORKDIR}/src:{WORKDIR}"
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

## 4. Install

Unsloth + Transformers v5 (required for the Qwen3.5/3.8 architecture). The project itself is installed with
`--no-deps`: this notebook only needs its data generator, schemas, prompts and metrics, and that keeps
Colab's preinstalled PyTorch untouched.

In [ ]:
%%capture
!pip install --upgrade unsloth unsloth_zoo
!pip install --upgrade "transformers>=5.0.0" trl datasets
!pip install --no-deps -e /content/fedsearch

In [ ]:
import importlib.metadata as md
for pkg in ("unsloth", "transformers", "trl", "torch", "datasets"):
    print(f"{pkg:13s}", md.version(pkg))

## 5. Generate the training data

The same synthetic, multi-system company used by the search engine. Every question has a gold `QueryPlan`.
The test split includes typos, nicknames ("okta login" → Single Sign-On) and phrasings never seen in training.

In [ ]:
import json, subprocess
from pathlib import Path
from fedsearch.seed.write import write_all

print(json.dumps(write_all(Path("data")), indent=1))
print(subprocess.run(["python", "training/build_sft.py"], capture_output=True, text=True, check=True).stdout)

example = json.loads(Path("data/sft/train.jsonl").read_text().splitlines()[0])
print("USER:     ", example["messages"][1]["content"])
print("ASSISTANT:", example["messages"][2]["content"])

## 6. Load the base model

In [ ]:
import unsloth  # must be imported before transformers / trl
from training.sft_lib import (TrainConfig, load_model, add_lora, load_sft_dataset, make_trainer,
                              generate_plans, score_plans, load_examples, save_adapter, parse_plan)

cfg = TrainConfig.from_yaml("configs/train_qlora.yaml", PROFILE)
print(cfg)
model, tokenizer = load_model(cfg)

## 7. Baseline: how good is the model *before* fine-tuning?

Zero-shot, same system prompt the search engine uses. An output that isn't valid JSON matching the schema
counts as fully wrong (`valid` = share of parseable plans).

In [ ]:
test = load_examples("data/queries/test.jsonl", EVAL_LIMIT)
before_raw = generate_plans(model, tokenizer, [e["query"] for e in test])
before = score_plans(test, before_raw)
print(json.dumps(before["overall"], indent=1))
print("\nexample output:", before_raw[0][:400])

## 8. Fine-tune (QLoRA)

Only the assistant turn (the JSON plan) contributes to the loss.

In [ ]:
model = add_lora(model, cfg)
ds = load_sft_dataset(tokenizer, cfg)
print(ds)
trainer = make_trainer(model, tokenizer, ds, cfg)
stats = trainer.train()
print(stats)

In [ ]:
import pandas as pd
logs = pd.DataFrame(trainer.state.log_history)
ax = logs.dropna(subset=["loss"]).plot(x="step", y="loss", label="train loss", figsize=(7, 3))
if "eval_loss" in logs:
    logs.dropna(subset=["eval_loss"]).plot(x="step", y="eval_loss", label="val loss", ax=ax)

## 9. After: the same test questions with the fine-tuned model

In [ ]:
after_raw = generate_plans(model, tokenizer, [e["query"] for e in test])
after = score_plans(test, after_raw)

METRICS = ["valid", "intent_acc", "plan_exact", "sources_exact", "entities_exact",
           "src_precision", "src_recall", "ent_precision", "ent_recall", "ent_f1"]
cmp = pd.DataFrame({"base": before["overall"], "fine-tuned": after["overall"]}).loc[METRICS]
cmp["Δ"] = cmp["fine-tuned"] - cmp["base"]
cmp.style.format("{:.3f}")

In [ ]:
groups = [k for k in after if k != "overall"]
by_group = pd.DataFrame({g: {"base": before[g]["plan_exact"], "fine-tuned": after[g]["plan_exact"],
                             "n": after[g]["n"]} for g in groups}).T
by_group[["base", "fine-tuned"]].plot.barh(figsize=(7, 4), title="Exact-plan accuracy by group", xlim=(0, 1))
by_group

In [ ]:
# Where does the fine-tuned model still go wrong?
for ex, raw in zip(test, after_raw):
    pred = parse_plan(raw)
    if pred is None or pred.intent != ex["plan"]["intent"] or \
       {(e.type, e.value.lower()) for e in pred.entities} != {(e["type"], e["value"].lower()) for e in ex["plan"]["entities"]}:
        print(f"Q: {ex['query']}\n  gold: {ex['plan']['intent']} {ex['plan']['entities']}\n  pred: {raw[:300]}\n")

## 10. Try your own questions

In [ ]:
for q in ["is globx still having trouble with okta login?",
          "what did folks say in chat about TCK-1042",
          "how do we fix E-GW-502 for a customer",
          "quick health check on wayne analytics"]:
    print(q, "\n ->", generate_plans(model, tokenizer, [q])[0], "\n")

## 11. Save the adapter and the evaluation report

In [ ]:
import shutil
out = Path(cfg.output_dir)
adapter_dir = save_adapter(model, tokenizer, out / "adapter")
report = {"profile": PROFILE, "gpu": gpu, "n_test": len(test), "before": before, "after": after,
          "train_runtime_s": stats.metrics.get("train_runtime")}
(out / "planner_eval.json").write_text(json.dumps(report, indent=2))
print("saved", adapter_dir)

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    dest = Path("/content/drive/MyDrive/fedsearch") / out.name
    dest.mkdir(parents=True, exist_ok=True)
    shutil.copytree(adapter_dir, dest / "adapter", dirs_exist_ok=True)
    shutil.copy(out / "planner_eval.json", dest / "planner_eval.json")
    print("copied to", dest)

if HF_REPO:
    model.push_to_hub(HF_REPO, token=_secret("HF_TOKEN"), private=True)
    tokenizer.push_to_hub(HF_REPO, token=_secret("HF_TOKEN"), private=True)
    print(f"pushed to https://huggingface.co/{HF_REPO}")

## 12. Next: serve it

Open **`notebooks/serve_vllm_colab.ipynb`**. It serves the base model + this adapter with vLLM behind an
OpenAI-compatible URL, which you point the search engine at from your laptop:

```bash
export FEDSEARCH_PLANNER=llm FEDSEARCH_LLM_BASE_URL=https://<tunnel>.trycloudflare.com/v1
uv run fedsearch eval        # full end-to-end retrieval + answer metrics with the fine-tuned planner
```

Optional — merged 16-bit weights (~55 GB, for serving without LoRA on an 80 GB GPU):
`model.save_pretrained_merged("merged", tokenizer, save_method="merged_16bit")`.